# Building molecules and crystals with ASE

This notebook introduces a few useful parts of the ASE [`build`](https://ase-lib.org/ase/build/build.html) and [`spacegroup`](https://ase-lib.org/ase/spacegroup/spacegroup.html) modules.  
The goal is twofold:

1. learn how to construct molecules and crystals directly from code;
2. see that reproducing a crystal structure from the literature is not always straightforward, even when a paper provides lattice parameters and fractional coordinates.

In several examples below, the data in the publication are enough to reconstruct a sensible structure. In other cases, a literal transcription of the reported numbers can lead to odd or even clearly unphysical geometries. This is not a failure of ASE. It is a useful reminder that one should always inspect the final structure carefully, verify stoichiometry, check the meaning of the reported coordinates, and compare with the discussion in the paper.


In [ ]:
%load_ext aiida
%aiida


## Imports and interactive viewer

We use ASE to build molecules and crystals, and an AiiDAlab-based structure manager to inspect the generated geometries interactively.


In [ ]:
from collections import Counter

import numpy as np
from numpy.linalg import norm

from ase.build import molecule
from ase.spacegroup import crystal

import aiidalab_widgets_base as awb
from IPython.display import display


In [ ]:
structure_selector = awb.StructureManagerWidget(
    importers=[
        awb.StructureUploadWidget(title="Import from computer"),
        awb.StructureBrowserWidget(title="AiiDA database"),
        awb.SmilesWidget(title="From SMILES"),
    ],
    editors=[
        awb.BasicStructureEditor(title="Edit structure"),
        awb.BasicCellEditor(),
    ],
    storable=True,
    node_class="StructureData",
)
display(structure_selector)


def show_structure(structure):
    """Send an ASE structure to the AiiDAlab viewer."""
    structure_selector.structure = structure
    return structure


def composition_dict(structure):
    """Return the atomic composition as a dictionary."""
    return dict(Counter(structure.get_chemical_symbols()))


def shortest_contacts(structure, n=10, mic=True):
    """Return the n shortest interatomic distances."""
    distances = structure.get_all_distances(mic=mic)
    symbols = structure.get_chemical_symbols()

    pairs = []
    for i in range(len(structure)):
        for j in range(i + 1, len(structure)):
            pairs.append((distances[i, j], i, j, symbols[i], symbols[j]))

    return sorted(pairs, key=lambda x: x[0])[:n]


def print_shortest_contacts(structure, n=10, mic=True):
    for d, i, j, si, sj in shortest_contacts(structure, n=n, mic=mic):
        print(f"{i:3d} {si:>2s} -- {j:3d} {sj:>2s} : {d:8.4f} Å")


## 1. Simple molecules

ASE can generate many standard molecules directly from a chemical formula or conventional name.  
This is convenient for quick model building and for checking basic geometric information such as bond lengths and angles.


In [ ]:
water = molecule("")
benzene = molecule("")
dmso = molecule("")


In [ ]:
show_structure(water)


In [ ]:
show_structure(benzene)


In [ ]:
show_structure(dmso)


For water, we can extract geometric information directly from the `Atoms` object.


In [ ]:
oh_1 = water.get_distance(0, 1)
oh_2 = water.get_distance(0, 2)
hoh = water.get_angle(1, 0, 2)

print(f"O-H distance 1: {oh_1:.4f} Å")
print(f"O-H distance 2: {oh_2:.4f} Å")
print(f"H-O-H angle   : {hoh:.2f}°")


## 2. Monoclinic HfO$_2$ from the literature

We now construct monoclinic hafnia using experimentally derived coordinates from:

- [Phys. Rev. B 65, 233106 (2002)](https://doi.org/10.1103/PhysRevB.65.233106)

This is a good example of how the `crystal` function combines:

- the **lattice parameters**,
- the **space group**,
- the **symmetry-inequivalent fractional coordinates**,

to generate the full crystal.


In [ ]:
a = 5.117
b = 5.175
c = 5.291
alpha = 90
beta = 
gamma = 90
spacegroup =

hfo2 = crystal(
    symbols="",
    basis=[
        (0.276, 0.040, 0.208),
        (0.074, 0.332, 0.347),
        (0.449, 0.758, 0.480),
    ],
    spacegroup=spacegroup,
    cellpar=[a, b, c, alpha, beta, gamma],
)


In [ ]:
print("Chemical formula :", hfo2.get_chemical_formula())
print("Composition      :", composition_dict(hfo2))
print("Number of atoms  :", len(hfo2))


In [ ]:
show_structure(hfo2)


Only a few coordinates are entered by hand, but the full crystal contains more atoms because ASE applies the space-group symmetry operations automatically. This is often the first point to check when reading a crystallographic table: are the reported coordinates the full structure, or only the asymmetric unit?


## 3. Orthorhombic MAPbI$_3$

Next we reconstruct the orthorhombic phase of MAPbI$_3$ using the supporting information of:

- [Chem. Commun. 51, 4180–4183 (2015)](https://doi.org/10.1039/C4CC09944C)

This example is useful because the labels in the table can be misleading if one reads them too quickly.  
Before doing anything expensive, it is always worth checking whether the resulting stoichiometry is what you expect.


In [ ]:
a = 8.86574
b = 12.6293
c = 8.57684
alpha = 90
beta = 90
gamma = 90

mapbi3_ortho = crystal(
    symbols="PbI2NCH4",  # some labels in the table can be misleading
    basis=[
        (0.5,    0.0,    0.0),
        (0.4842, 0.25,  -0.0562),
        (0.1886, 0.0147, 0.1844),
        (0.9421, 0.75,   0.0297),
        (0.9372, 0.25,   0.0575),
        (0.9372, 0.25,   0.1874),
        (0.8661, 0.1701, 0.0290),
        (0.1275, 0.1891,-0.0085),
        (0.9543, 0.75,   0.1459),
    ],
    spacegroup=62,
    cellpar=[a, b, c, alpha, beta, gamma],
)


In [ ]:
print("Chemical formula :", mapbi3_ortho.get_chemical_formula())
print("Composition      :", composition_dict(mapbi3_ortho))
print("Number of atoms  :", len(mapbi3_ortho))


In [ ]:
show_structure(mapbi3_ortho)


In [ ]:
mapbi3_supercell = mapbi3_ortho.repeat((2, 2, 2))
print("Supercell formula:", mapbi3_supercell.get_chemical_formula())
print("Number of atoms  :", len(mapbi3_supercell))


In [ ]:
show_structure(mapbi3_supercell)


## 4. Cubic MAPbI$_3$: when the published information is not enough

Using the same article, we now try to construct the cubic phase directly from the reported data.
The interesting part here is not just whether ASE can build *something*, but whether the resulting structure is chemically sensible.

This is the point where visualization and distance checks become essential.  
A crystal can satisfy the formal input requirements of `crystal(...)` and still look wrong because of disorder, missing orientational information, averaging in the experimental model, transcription mistakes, or a mismatch between what is tabulated and what a static atomistic model requires.


In [ ]:
a = 6.317228
b = a
c = a
alpha = 90
beta = 90
gamma = 90

mapbi3_cubic = crystal(
    symbols="PbICH",
    basis=[
        (0.0,      0.0,      0.0),
        (0.5,      0.0,      0.0),
        (0.393308, 0.500000, 0.500000),
        (0.305848, 0.448981, 0.379518),
    ],
    spacegroup=221,
    cellpar=[a, b, c, alpha, beta, gamma],
)


In [ ]:
print("Chemical formula :", mapbi3_cubic.get_chemical_formula())
print("Composition      :", composition_dict(mapbi3_cubic))
print("Number of atoms  :", len(mapbi3_cubic))
print()
print("Shortest contacts:")
print_shortest_contacts(mapbi3_cubic, n=10)


In [ ]:
show_structure(mapbi3_cubic)


If the geometry looks suspicious, that is exactly the point of the exercise.  
The lesson is that one should not blindly trust a direct transcription of a table into code. In some materials,  the experimentally reported structure can represent an averaged or disordered picture rather than a unique static molecular arrangement. In such cases, a naïve reconstruction may produce nonsense contacts or an incomplete molecular fragment.


## 5. Low-temperature Ta$_2$O$_5$

This final example highlights a different practical issue: extracting structural data from the literature is sometimes error-prone even before the model is built.  
If coordinates are transcribed manually, or extracted from table images with OCR or AI tools, one must check:

- whether the caption refers to the correct phase,
- whether the fractional coordinates were copied correctly,
- whether the correct setting and space group were used,
- whether the resulting structure is chemically meaningful.

Comparing the shortest interatomic distances is a simple and effective sanity check.


In [ ]:
coordinates = [
    (0.4752, 0.86660, 0.0),   # Ta1
    (0.4731, 0.92219, 0.0),   # Ta2
    (0.9835, 0.78908, 0.0),   # Ta3
    (0.0,    0.0,     0.0),   # Ta4
    (0.9179, 0.89520, 0.0),   # Ta5
    (0.4308, 0.81472, 0.0),   # Ta6
    (0.4511, 0.97513, 0.0),   # Ta7
    (0.0022, 0.94516, 0.0),   # Ta8
    (0.5247, 0.76499, 0.0),   # Ta9
    (0.9859, 0.84435, 0.0),   # Ta10

    (0.697, 0.8308, 0.0),     # O1
    (0.563, 0.8948, 0.0),     # O2
    (0.335, 0.9475, 0.0),     # O3
    (0.686, 0.9930, 0.0),     # O4
    (0.079, 0.8170, 0.0),     # O5
    (0.802, 0.9221, 0.0),     # O6
    (0.099, 0.9743, 0.0),     # O7
    (0.861, 0.7642, 0.0),     # O8
    (0.306, 0.7866, 0.0),     # O9
    (0.320, 0.8425, 0.0),     # O10
    (0.170, 0.9161, 0.0),     # O11
    (0.804, 0.8680, 0.0),     # O12
    (0.924, 0.8940, 0.5),     # O13
    (0.525, 0.7659, 0.5),     # O14
    (0.170, 0.8772, 0.0),     # O15
    (0.717, 0.9583, 0.0),     # O16
    (0.25,  0.75,   0.0),     # O17
    (0.675, 0.7949, 0.0),     # O18
    (0.992, 0.8449, 0.5),     # O19
    (0.491, 0.8656, 0.5),     # O20
    (0.010, 0.9441, 0.0),     # O21
    (0.0,   0.0,    0.5),     # O22
    (0.436, 0.8153, 0.5),     # O23
    (0.996, 0.7883, 0.5),     # O24
    (0.454, 0.9757, 0.5),     # O25
    (0.486, 0.9231, 0.5),     # O26
]


In [ ]:
a = 6.1939
b = 69.549
c = 3.8895

ta2o5 = crystal(
    symbols="Ta10O26",
    basis=coordinates,
    spacegroup=11,
    cellpar=[a, b, c, 90, 90, 90],
    setting=1,
)


In [ ]:
print("Chemical formula :", ta2o5.get_chemical_formula())
print("Composition      :", composition_dict(ta2o5))
print("Number of atoms  :", len(ta2o5))
print()
print("Shortest contacts:")
print_shortest_contacts(ta2o5, n=12)


In [ ]:
show_structure(ta2o5)


## Final remarks

A few habits are worth developing whenever you build a crystal structure from the literature:

- inspect the geometry visually;
- verify the stoichiometry;
- check whether the tabulated coordinates correspond to the full structure or only the asymmetric unit;
- compute a few short interatomic distances to catch obviously wrong geometries;
- read the surrounding discussion in the paper, not just the table.

The main point is not only to learn the ASE syntax, but also to learn how to *doubt* a structural model.
